# Records and Record Distributions

*If you haven't read the [Getting Started tutorial](../tutorials/getting_started.ipynb), read it first — it introduces the three core objects in ProbPipe (`Distribution`, `Record`, `WorkflowFunction`) and the workflow they fit into.*

ProbPipe applies one design idea twice — once to *non-random* values (the Record family) and once to *random* quantities (the RecordDistribution family). Both expose the same named-field surface, both flow through `@workflow_function` broadcasting the same way, both round-trip through JAX pytrees. This notebook walks both sides in parallel.

The shape of the parallel:

|                       | Non-random (value)        | Random (one variable)               |
|-----------------------|---------------------------|-------------------------------------|
| **Permissive leaves** | `Record`                  | `RecordDistribution`                |
| **Numeric leaves**    | `NumericRecord`           | `NumericRecordDistribution`         |

Rows are about what can live in a leaf (any Python object vs `jax.Array`); columns are about whether the thing is fixed or random. Each side also has an *array of* form for batches:

- Value side: `RecordArray` (with `NumericRecordArray` specialisation).
- Distribution side: `DistributionArray`.

The two array forms are not perfectly symmetric: `RecordArray` *is* a `Record` whose leaves happen to be batched, but `DistributionArray` is *not* a `RecordDistribution` — it's a positional collection of independent scalar distributions, a different concept. We'll cover both at the end.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from probpipe import (
    Record,
    NumericRecord,
    RecordArray,
    NumericRecordArray,
    RecordTemplate,
    NumericRecordTemplate,
    # Distribution-side
    Normal,
    Bernoulli,
    Categorical,
    MultivariateNormal,
    ProductDistribution,
    DistributionArray,
    NumericRecordDistribution,
    sample,
    WorkflowKind,
    prefect_config,
)

# This notebook walks Records and Distributions, not orchestration —
# disable the Prefect path so ``sample`` etc. run synchronously.
prefect_config.workflow_kind = WorkflowKind.OFF

# Part I — Values: the Record family

## 1. `Record` basics

Construct a `Record` by passing named fields. Leaves can be anything —
arrays, scalars, strings, xarray objects, or nested Records. Nothing is
coerced at construction time: each leaf is stored verbatim.

In [2]:
params = Record(growth_rate=1.8, carrying_capacity=70.0)
print(params)
print("fields:", params.fields)

Record(growth_rate=1.8, carrying_capacity=70.0)
fields: ('growth_rate', 'carrying_capacity')


Fields iterate in **insertion order** — the same order the keyword arguments were passed (or the order of the input dict). Access fields with `record["name"]` — not attribute access:


In [3]:
print("growth_rate:", params["growth_rate"])
print("carrying_capacity:", params["carrying_capacity"])

growth_rate: 1.8
carrying_capacity: 70.0


Attribute access deliberately doesn't work. This separation keeps your
field names from ever colliding with the API surface:

In [4]:
try:
    params.growth_rate
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'Record' object has no attribute 'growth_rate'


The `Record` itself has the usual dict-like protocol: `len`, `in`, iteration,
`.keys()`, `.values()`, `.items()`.

In [5]:
print("len:", len(params))
print("growth_rate in params:", "growth_rate" in params)
for name, val in params.items():
    print(f"  {name} = {val}")

len: 2
growth_rate in params: True
  growth_rate = 1.8
  carrying_capacity = 70.0


### Heterogeneous leaves

Because `Record` stores leaves verbatim, fields can mix types freely:

In [6]:
observation = Record(counts=np.array([2, 1, 3, 0, 5]), label='horseshoe', duration=10.0)
print(observation)
print("counts:", observation["counts"])
print("label:", observation["label"])

Record(counts=array(shape=(5,)), label='horseshoe', duration=10.0)
counts: [2 1 3 0 5]
label: horseshoe


### Immutability

A `Record` is immutable. You can't reassign or delete a field:

In [7]:
try:
    params["growth_rate"] = 2.0
except TypeError as e:
    print("TypeError:", e)

TypeError: 'Record' object does not support item assignment


Instead, you build new Records from old ones with `replace`, `merge`,
`without`. Each returns a new Record of the same class as `self`, so the
operation is lossless for `NumericRecord` subclasses too.

In [8]:
params2 = params.replace(growth_rate=2.0)
print("original:", params)
print("replaced:", params2)

original: Record(growth_rate=1.8, carrying_capacity=70.0)
replaced: Record(growth_rate=2.0, carrying_capacity=70.0)


In [9]:
extra = Record(phi=10.0)
combined = params.merge(extra)
print("merged:", combined)

merged: Record(growth_rate=1.8, carrying_capacity=70.0, phi=10.0)


In [10]:
reduced = combined.without("phi")
print("reduced:", reduced)

reduced: Record(growth_rate=1.8, carrying_capacity=70.0)


## 2. Nested Records

Fields can themselves be Records. This is useful when you want to group
related quantities — e.g., separating physical parameters from observation
noise in a likelihood.

In [11]:
nested = Record(physics=Record(force=1.2, mass=3.5), noise=Record(sigma=0.1))
print(nested)

Record(physics=Record(force=1.2, mass=3.5), noise=Record(sigma=0.1))


Top-level access uses the string key as usual:

In [12]:
print("physics block:", nested["physics"])

physics block: Record(force=1.2, mass=3.5)


For nested leaves, either chain bracket accesses or pass a tuple key — the
tuple form is a shortcut that reads left-to-right:

In [13]:
print("via chaining:", nested["physics"]["force"])
print("via tuple:   ", nested["physics", "force"])

via chaining: 1.2
via tuple:    1.2


`jax.tree.map` walks through nested Records automatically (they're registered
as pytrees):

In [14]:
doubled = jax.tree.map(lambda x: x * 2.0, nested)
print("doubled:", doubled)

doubled: Record(physics=Record(force=2.4, mass=7.0), noise=Record(sigma=0.2))


## 3. `NumericRecord` — the numeric-leaves specialisation

`NumericRecord` is a `Record` subclass that validates every leaf at
construction time: each must be a numeric array, a numeric scalar, or a
nested `NumericRecord`. Non-numeric leaves raise `TypeError`. Numeric
leaves are coerced to `jnp.ndarray` so downstream JAX code sees a uniform
array type.

In [15]:
theta = NumericRecord(r=1.8, K=70.0, phi=10.0)
print(theta)
print("r is jnp array:", isinstance(theta["r"], jnp.ndarray))

NumericRecord(r=Array(1.8, dtype=float32, weak_type=True), K=Array(70., dtype=float32, weak_type=True), phi=Array(10., dtype=float32, weak_type=True))
r is jnp array: True


The validation catches non-numeric leaves immediately — you can't
accidentally mix a label string in with your numeric parameters:

In [16]:
try:
    NumericRecord(r=1.8, label="horseshoe")
except TypeError as e:
    print("TypeError:", e)

TypeError: NumericRecord: field 'label' must be a numeric array, numeric scalar, or nested NumericRecord, got str


### `flatten` and `unflatten`

Because every leaf is a `jax.Array`, `NumericRecord` gains a lossless flat-vector representation. `flatten()` concatenates every leaf's raveled values in field-insertion order; `unflatten(flat, template=...)` reverses the process.


In [17]:
params = NumericRecord(r=1.8, K=jnp.array([70.0, 72.0]), phi=10.0)
flat = params.flatten()
print("flat_size:", params.flat_size)
print("flat vector:", flat)

flat_size: 4
flat vector: [ 1.8 70.  72.  10. ]


`flat_size` is cached at construction and matches `len(flatten())`. It's
the number of scalar elements across every numeric leaf — useful when
feeding the Record into an algorithm that wants a fixed-dimensional input
(MCMC samplers, optimisers, neural nets).

## 4. `RecordTemplate` / `NumericRecordTemplate` — the structural skeleton

`unflatten` needs to know the original field names and per-field shapes,
since those are erased by flattening. `RecordTemplate` carries that
structural information with no data attached. There are two closely
related classes:

- **`RecordTemplate`** — the general case. Each spec is either a shape
  tuple (numeric leaf), `None` (opaque leaf — string, label, any
  non-array data), or a nested sub-template. Exposes `leaf_shapes`.
- **`NumericRecordTemplate`** — the all-numeric specialisation. Rejects
  `None` specs outright, so every leaf is a shape tuple or a nested
  `NumericRecordTemplate`. Adds `flat_size` (total scalar count) and
  `numeric_leaf_shapes`, which are only meaningful when there's nothing
  opaque to skip.

`RecordTemplate(...)` auto-promotes to `NumericRecordTemplate` when every
spec happens to be numeric, so the common case works without naming the
subclass.

In [18]:
template = RecordTemplate.from_record(params)
print("template:", template)
print("type:", type(template).__name__)
print("leaf shapes:", template.leaf_shapes)
print("flat_size:", template.flat_size)

template: NumericRecordTemplate(r=(), K=(2,), phi=())
type: NumericRecordTemplate
leaf shapes: {'r': (), 'K': (2,), 'phi': ()}
flat_size: 4


Round-trip:

In [19]:
restored = NumericRecord.unflatten(flat, template=template)
print("restored:", restored)
print("equal to original:", restored == params)

restored: NumericRecord(r=Array(1.8, dtype=float32), K=array(shape=(2,)), phi=Array(10., dtype=float32))
equal to original: True


You can also build a template directly, without an example Record — useful
when describing the expected structure of a distribution's sample before
any draw has happened:

In [20]:
tpl = RecordTemplate(r=(), K=(2,), phi=())
print("direct template:", tpl)
print("type:", type(tpl).__name__, "(auto-promoted — all specs numeric)")
print("flat_size:", tpl.flat_size)

direct template: NumericRecordTemplate(r=(), K=(2,), phi=())
type: NumericRecordTemplate (auto-promoted — all specs numeric)
flat_size: 4


### Mixed leaves — why `flat_size` lives on the numeric template

When a template contains any opaque (`None`) leaves, it stays a plain
`RecordTemplate`. Opaque leaves don't carry a scalar count, so there
isn't a single "correct" interpretation of `flat_size` (is an opaque
leaf zero elements? one? a placeholder for something that will be
numeric later?). Rather than pick silently, the base class simply
doesn't define `flat_size` or `numeric_leaf_shapes` — those are only
defined on `NumericRecordTemplate`, where every leaf is numeric and
the count is unambiguous.

In [21]:
mixed_tpl = RecordTemplate(counts=(5,), label=None)
print("mixed template:", mixed_tpl)
print("type:", type(mixed_tpl).__name__, "(stays base — has an opaque leaf)")
print("leaf_shapes:", mixed_tpl.leaf_shapes)

# flat_size / numeric_leaf_shapes aren't defined on the mixed template.
for attr in ("flat_size", "numeric_leaf_shapes"):
    print(f"  has {attr}:", hasattr(mixed_tpl, attr))

# Trying to build a NumericRecordTemplate with an opaque leaf is rejected.
try:
    NumericRecordTemplate(counts=(5,), label=None)
except TypeError as e:
    print("NumericRecordTemplate rejects None:", e)

mixed template: RecordTemplate(counts=(5,), label=None)
type: RecordTemplate (stays base — has an opaque leaf)
leaf_shapes: {'counts': (5,), 'label': None}
  has flat_size: False
  has numeric_leaf_shapes: False
NumericRecordTemplate rejects None: NumericRecordTemplate: field 'label' is opaque (spec=None); opaque leaves are not allowed — use RecordTemplate if you need a mixed template.


### Nested templates

Templates compose: the spec for a field can be a tuple (numeric leaf),
`None` (opaque), or another template (nested structure). Auto-promotion
propagates — if every nested sub-template is numeric and the top-level
specs are all numeric, the whole tree becomes a
`NumericRecordTemplate`.

In [22]:
nested_tpl = RecordTemplate(physics=RecordTemplate(force=(), mass=()), obs=(5,))
print("nested template:", nested_tpl)
print("type:", type(nested_tpl).__name__)
print("flat_size:", nested_tpl.flat_size, "(2 scalars + 5-vector)")
print("flat leaf shapes:", nested_tpl.leaf_shapes)

nested template: NumericRecordTemplate(physics=NumericRecordTemplate(force=(), mass=()), obs=(5,))
type: NumericRecordTemplate
flat_size: 7 (2 scalars + 5-vector)
flat leaf shapes: {'physics/force': (), 'physics/mass': (), 'obs': (5,)}


## 5. `RecordArray` — batches of Records

A `RecordArray` stores a batch of Records with shared field structure.
Each field has shape `(*batch_shape, *leaf_shape)`. Typical use case: a
collection of MCMC draws, a batch of simulated datasets, a grid of
parameter proposals.

Construct from a list of Records via `.stack()`:

In [23]:
draws = [NumericRecord(r=1.8 + 0.1 * i, K=70.0 + i, phi=10.0) for i in range(4)]
batch = NumericRecordArray.stack(draws)
print("batch_shape:", batch.batch_shape)
print("template:", batch.template)
print(batch)

batch_shape: (4,)
template: NumericRecordTemplate(r=(), K=(), phi=())
NumericRecordArray(batch_shape=(4,), r=array(shape=(4,)), K=array(shape=(4,)), phi=array(shape=(4,)))


### Two kinds of indexing

**String key** → returns the batched leaf array (shape
`(*batch_shape, *leaf_shape)`):

In [24]:
print("batch['r']:", batch["r"])
print("  shape:", batch["r"].shape)

batch['r']: [1.8 1.9 2.  2.1]
  shape: (4,)


**Integer index** → returns the single element at that flat batch position,
materialised as a `Record` (or, for `NumericRecordArray`, a `NumericRecord`):

In [25]:
print("batch[2]:", batch[2])
print("  type:", type(batch[2]).__name__)

batch[2]: NumericRecord(r=Array(2., dtype=float32, weak_type=True), K=Array(72., dtype=float32, weak_type=True), phi=Array(10., dtype=float32, weak_type=True))
  type: NumericRecord


These two indexing modes never collide because Records use strings for
fields and arrays use ints for positions. You can use either without
thinking about which dimension you meant.

### `NumericRecordArray` — batched numeric operations

`NumericRecordArray` is a `RecordArray` subclass with two extra guarantees
and two extra operations. The guarantees are that every leaf must have a
numeric dtype and shape `(*batch_shape, *event_shape)`; construction
validates both. The extra operations are `flatten` / `unflatten` and
reductions.

In [26]:
print("mean across draws:", batch.mean())
print("  type:", type(batch.mean()).__name__)
print("variance across draws:", batch.var())

mean across draws: NumericRecord(r=Array(1.9499999, dtype=float32), K=Array(71.5, dtype=float32), phi=Array(10., dtype=float32))
  type: NumericRecord


variance across draws: NumericRecord(r=Array(0.0125, dtype=float32), K=Array(1.25, dtype=float32), phi=Array(0., dtype=float32))


`mean` returns a `NumericRecord` when reduction eliminates all batch
dimensions, otherwise a smaller `NumericRecordArray`. `.flatten()`
concatenates every field into a trailing axis, preserving `batch_shape`:

In [27]:
flat_batch = batch.flatten()
print("flattened shape:", flat_batch.shape)  # (4, 3) = (batch_size, flat_event_size)
print(flat_batch)

flattened shape: (4, 3)
[[ 1.8 70.  10. ]
 [ 1.9 71.  10. ]
 [ 2.  72.  10. ]
 [ 2.1 73.  10. ]]


Round-tripping through the flat representation works the same way as for
a single `NumericRecord`, but now the template describes the event
structure and the batch dimensions are preserved automatically:

In [28]:
restored_batch = NumericRecordArray.unflatten(
    flat_batch, template=batch.template, batch_shape=batch.batch_shape,
)
print("restored:", restored_batch)
print("equal:", restored_batch == batch)

restored: NumericRecordArray(batch_shape=(4,), r=array(shape=(4,)), K=array(shape=(4,)), phi=array(shape=(4,)))
equal: True


If you omit `batch_shape`, `unflatten` infers it from `flat.shape[:-1]`:

In [29]:
inferred = NumericRecordArray.unflatten(flat_batch, template=batch.template)
print("inferred batch_shape:", inferred.batch_shape)

inferred batch_shape: (4,)


### Why is `RecordArray` not hashable?

`Record` and `NumericRecord` are hashable — their `__hash__` uses the
class name, field names, and per-leaf shape+dtype, so two Records with
the same structure hash the same even though they may hold different
values. That's cheap and matches how JAX / tree-util use structural
equality.

`RecordArray` is deliberately unhashable, following NumPy's precedent:
hashing by value would require materialising the whole batch (huge for
posterior draws), and it would fail inside `jit` / `vmap` because traced
arrays aren't hashable by content. If you need a structural key, use
`(type(ra), ra.batch_shape, ra.fields, ra.template)`.

# Part II — Random: the RecordDistribution family

## 6. `RecordDistribution` — the parallel concept

A `RecordDistribution` is what `Record` becomes when the values are random. The named-field surface — `.fields`, `dist[name]`, `dist.select_all()` — is identical, just operating on random components.

You don't instantiate `RecordDistribution` directly. You construct one of its concrete flavours (`ProductDistribution`, `SequentialJointDistribution`, `JointGaussian`, `JointEmpirical`), each a different way to build the joint. The walks for those four live in [the joint distributions notebook](04_joint_distributions.ipynb); here we use `ProductDistribution` for the surface demo.

In [30]:
# Two objects with the same field names — one fixed, one random.
fixed = Record(growth_rate=1.8, carrying_capacity=70.0)
prior = ProductDistribution(
    growth_rate=Normal(loc=0.0, scale=3.0, name="growth_rate"),
    carrying_capacity=Normal(loc=70.0, scale=10.0, name="carrying_capacity"),
)

print("fields:")
print(f"  Record:               {fixed.fields}")
print(f"  RecordDistribution:   {prior.fields}")

print("\nindexing:")
print(f"  Record[name]:               {fixed['growth_rate']}")
print(f"  RecordDistribution[name]:   {type(prior['growth_rate']).__name__}  (a view)")

print("\nselect_all keys:")
print(f"  Record:               {list(fixed.select_all().keys())}")
print(f"  RecordDistribution:   {list(prior.select_all().keys())}")

fields:


  Record:               ('growth_rate', 'carrying_capacity')
  RecordDistribution:   ('growth_rate', 'carrying_capacity')

indexing:
  Record[name]:               1.8
  RecordDistribution[name]:   _RecordDistributionView  (a view)

select_all keys:
  Record:               ['growth_rate', 'carrying_capacity']
  RecordDistribution:   ['growth_rate', 'carrying_capacity']


The view returned by `dist[name]` is a lightweight wrapper that carries its parent reference. Two views from the same parent stay correlated when they're passed as siblings to a `@workflow_function`: the sweep layer samples the parent joint once per draw and extracts each component from that shared draw. The [broadcasting notebook](03_broadcasting.ipynb) covers the mechanism.

## 7. `NumericRecordDistribution` — the numeric specialisation

`NumericRecordDistribution` mirrors `NumericRecord`: every field is a numeric random variable. Standard distributions (`Normal`, `Beta`, `Bernoulli`, ...) inherit from `NumericRecordDistribution` via `TFPDistribution`, so any scalar TFP-backed class is a single-field `NumericRecordDistribution` under the hood.

Each numeric piece of metadata comes in a **canonical / convenience** pair:

| Concept   | Canonical (per-field, multi-leaf safe) | Convenience (single-field, scalar) |
|-----------|----------------------------------------|------------------------------------|
| Shape     | `event_shapes : dict`                  | `event_shape : tuple`              |
| Dtype     | `dtypes : dict`                        | `dtype` (unique value or `None`)   |
| Support   | `supports : dict`                      | `support : Constraint`             |

Subclasses override the canonical side; the scalar convenience derives.

In [31]:
n = Normal(loc=0.0, scale=1.0, name="x")

# Canonical (per-field).
print("event_shapes:", n.event_shapes)
print("dtypes:      ", n.dtypes)
print("supports:    ", n.supports)

# Convenience (single-field shortcut).
print("\nevent_shape:", n.event_shape)
print("dtype:       ", n.dtype)
print("support:     ", n.support)

event_shapes: {'x': ()}
dtypes:       {'x': <class 'numpy.float32'>}
supports:     {'x': real}

event_shape: ()
dtype:        <class 'numpy.float32'>
support:      real


Integer-valued distributions report integer dtypes — not the silent float default that earlier versions returned:

In [32]:
print("Bernoulli.dtype:   ", Bernoulli(probs=0.5, name="x").dtype)
print("Categorical.dtype: ", Categorical(probs=jnp.array([0.5, 0.5]), name="x").dtype)
print("Normal.dtype:      ", Normal(loc=0.0, scale=1.0, name="x").dtype)

Bernoulli.dtype:    <class 'jax.numpy.int32'>
Categorical.dtype:  <class 'jax.numpy.int32'>
Normal.dtype:       <class 'numpy.float32'>


A `name=` and an `event_shape` are enough to auto-build a single-field `RecordTemplate`. Standard distributions get this for free; you almost never construct the template by hand.

In [33]:
mvn = MultivariateNormal(loc=jnp.zeros(3), cov=jnp.eye(3), name="z")
print("record_template:", mvn.record_template)
print("event_shapes:   ", mvn.event_shapes)

record_template: NumericRecordTemplate(z=(3,))
event_shapes:    {'z': (3,)}


## 8. Samples bridge to the value side

What does `sample(dist)` return? It depends on how many fields the distribution has:

- **Single-field** (`Normal`, `MultivariateNormal`, ...) → raw `jax.Array`   shaped `sample_shape + event_shape`.
- **Multi-field** (`ProductDistribution`, ...) → `Record` (or   `NumericRecordArray` for non-empty `sample_shape`), keyed by `dist.fields`.

The boundary is set by `record_template`: single-leaf templates emit arrays, multi-leaf emit Records. The `treedef` property locks this relationship — it derives from the template, so a multi-leaf distribution's samples always have the same pytree shape.

In [34]:
# Single-field → array.
n = Normal(loc=0.0, scale=1.0, name="x")
print("Normal sample type:    ", type(sample(n, key=jax.random.PRNGKey(0))).__name__)

# Multi-field → Record.
prior = ProductDistribution(
    growth_rate=Normal(loc=0.0, scale=3.0, name="growth_rate"),
    carrying_capacity=Normal(loc=70.0, scale=10.0, name="carrying_capacity"),
)
draw = sample(prior, key=jax.random.PRNGKey(0))
print("ProductDistribution sample type:", type(draw).__name__)
print("  fields:", draw.fields)
print("  growth_rate:", float(draw['growth_rate']))

Normal sample type:     NumericRecord
ProductDistribution sample type: Record
  fields: ('growth_rate', 'carrying_capacity')
  growth_rate: 3.012042760848999


# Part III — Tying the two families together

## 9. The value ↔ distribution correspondence

The 2×2 from the opener, with concrete examples:

|                       | Non-random (value)                                       | Random (one variable)                                          |
|-----------------------|----------------------------------------------------------|----------------------------------------------------------------|
| **Permissive leaves** | `Record(label='horseshoe', count=5)`                     | `ProductDistribution(theta=Normal(0, 1), label=...)` (mixed)   |
| **Numeric leaves**    | `NumericRecord(r=1.8, K=70.0)`                           | `ProductDistribution(r=Normal(...), K=Normal(...))`            |

`select_all()` is the cleanest evidence the parallel is real — the same splat pattern works on either side. When `theta` is fixed, the function runs once with concrete values; when it's a distribution, the workflow function layer samples and MC-marginalises automatically.

In [35]:
def predict(growth_rate, carrying_capacity, x):
    return carrying_capacity / (1 + jnp.exp(-growth_rate * x))

x = jnp.linspace(-2.0, 2.0, 3)

# Fixed Record → one concrete evaluation.
fixed = NumericRecord(growth_rate=1.8, carrying_capacity=70.0)
print("predict on fixed values:", predict(**fixed.select_all(), x=x))

# Random RecordDistribution → same call signature; one draw demoed here.
prior = ProductDistribution(
    growth_rate=Normal(loc=0.0, scale=3.0, name="growth_rate"),
    carrying_capacity=Normal(loc=70.0, scale=10.0, name="carrying_capacity"),
)
draw = sample(prior, key=jax.random.PRNGKey(0))
print("predict on one prior draw:", predict(**draw.select_all(), x=x))

predict on fixed values: [ 1.8617897 35.        68.138214 ]


predict on one prior draw: [ 0.11001558 22.787722   45.46543   ]


`len` and iteration follow consistent conventions across the family:

- `len(record)` — number of fields.
- `len(record_array)` — leading-axis batch size (matches `np.zeros(batch_shape).__len__`).
- `len(distribution_array)` — same as `RecordArray`: leading-axis batch size.

The Record family iterates field names dict-style; `DistributionArray` iterates leading-axis slices like a numpy array.

## 10. `DistributionArray` — array of distributions

`DistributionArray` is the distribution-side counterpart to `RecordArray`: it holds a shape-indexed collection of independent scalar distributions.

The asymmetry: `RecordArray` *is* a `Record` whose leaves are batched, but `DistributionArray` is *not* a `RecordDistribution` — it's a positional collection where each cell is its own scalar `Distribution`. Reach for it when you need an array of distributions; reach for a `RecordDistribution` when you need one random variable with named components.

The natural way to construct one is `from_batched_params`, which dispatches on the per-class fused storage protocol:

In [36]:
da = DistributionArray.from_batched_params(
    Normal, loc=jnp.arange(5.0), scale=1.0, name="x",
)
print("DistributionArray:", da)
print("  batch_shape:", da.batch_shape)
print("  size:       ", da.size)
print("  len:        ", len(da))
print("  da[2].loc:  ", float(da[2].loc))

DistributionArray: DistributionArray(batch_shape=(5,), event_shape=() backend=True)
  batch_shape: (5,)
  size:        5
  len:         5
  da[2].loc:   2.0


The [broadcasting notebook](03_broadcasting.ipynb) covers how a `DistributionArray` flows through `@workflow_function` calls (cell-by-cell dispatch); [the joint distributions notebook](04_joint_distributions.ipynb) covers the four `RecordDistribution` flavours in depth.

`jax.tree.map` walks Records, NumericRecords, and Record-valued samples from joint distributions identically — they're all registered as pytrees with field names as aux data and leaves as children.

## 11. JAX interop carries through both sides

All four Record classes are registered as JAX pytrees with the field
names as aux data and the leaves as children. This means they flow
transparently through every JAX transform.

In [37]:
# jax.tree.map walks into the Record
theta = NumericRecord(r=1.8, K=70.0)
print("log params:", jax.tree.map(jnp.log, theta))

log params: NumericRecord(r=Array(0.5877866, dtype=float32, weak_type=True), K=Array(4.248495, dtype=float32, weak_type=True))


`jit` works because Records are pytrees — JAX knows how to trace through
them. The static structure (field names) becomes aux data, so two Records
with the same field names share a traced function:

In [38]:
@jax.jit
def logistic(theta):
    """Population at equilibrium given logistic parameters."""
    return theta["K"] * jnp.exp(theta["r"])

print("logistic(theta) =", logistic(theta))

logistic(theta) = 423.4753


Running the same jitted function on a `NumericRecordArray` broadcasts
automatically — every field is already a batched array, so JAX vectorises
the arithmetic without any extra effort:

In [39]:
theta_batch = NumericRecordArray.stack([NumericRecord(r=0.1 * i, K=70.0 + i) for i in range(5)])
print("logistic(theta_batch):", logistic(theta_batch))

logistic(theta_batch): [ 70.       78.46714  87.941    98.53969 110.39502]


## 12. Round-tripping xarray and pandas

ProbPipe's native array form is the JAX array, but it's common to build a `Record` from richer backends — `xarray.DataArray` with dims and coords, `pandas.Series` with a `DatetimeIndex`. The aux registry in `probpipe.core._array_backend` captures backend-specific metadata so a `Record` → `NumericRecord` → `Record` round-trip restores the original objects.


In [40]:
import xarray as xr

da = xr.DataArray(
    np.array([[1.0, 2.0], [3.0, 4.0]]),
    dims=("time", "species"),
    coords={"time": [0.0, 1.0], "species": ["fox", "hare"]},
    name="counts",
)
obs = Record(counts=da)

# Forward: convert to JAX-native form. Aux is captured per leaf.
numeric = obs.to_numeric()  # NumericRecord — every leaf is jax.Array
print("numeric[counts].dtype:", numeric["counts"].dtype)
print("numeric.aux keys:    ", list((numeric.aux or {}).keys()))

# Reverse: restore each leaf to its original backend.
back = numeric.to_native()
print("back[counts]:")
print(back["counts"])


numeric[counts].dtype: float32
numeric.aux keys:     ['counts']
back[counts]:
<xarray.DataArray 'counts' (time: 2, species: 2)> Size: 16B
array([[1., 2.],
       [3., 4.]], dtype=float32)
Coordinates:
  * time     (time) float64 16B 0.0 1.0
  * species  (species) <U4 32B 'fox' 'hare'


Aux is a snapshot at construction. Any transform that rebuilds the record (`record.map`, `record.replace`, `jax.tree.map`) drops it, and the transformed leaves come back as plain `jax.Array`. `probpipe.register_aux(...)` extends the registry with additional backend types beyond the built-in xarray / pandas registrations.


## Summary

| Need | Reach for |
|---|---|
| One named, immutable, non-random struct | `Record` |
| Same, numeric leaves, with `flatten` / single-field array shim | `NumericRecord` |
| Batch of those | `RecordArray` / `NumericRecordArray` |
| One random variable with named components | A concrete `RecordDistribution` — `ProductDistribution`, `JointGaussian`, `JointEmpirical`, `SequentialJointDistribution` |
| Same, numeric, with canonical `event_shapes` / `dtypes` / `supports` | A concrete `NumericRecordDistribution` — almost always a TFP-backed class like `Normal`, `Beta`, ... |
| Array of independent scalar distributions | `DistributionArray.from_batched_params(...)` (or as a WF-sweep output) |

The next step is `@workflow_function` and broadcasting in the [broadcasting notebook](03_broadcasting.ipynb), which is how Records and RecordDistributions get threaded together into a pipeline. The four concrete `RecordDistribution` flavours have their own walks in the [joint distributions notebook](04_joint_distributions.ipynb).